# Token evolution

A small genetic search evolves token sequences whose CLIP text embeddings resemble a target image. CUDA is required.

In [ ]:
import random
import torch
from diffusers.utils import load_image
from transformers import CLIPModel, CLIPProcessor

# Small defaults make the example easier to inspect.
image_url = "https://picsum.photos/seed/target/512"
n_tokens, population, generations = 8, 128, 20
seed = 0

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required.")

random.seed(seed)
torch.manual_seed(seed)
model_id = "openai/clip-vit-large-patch14"
model = CLIPModel.from_pretrained(model_id, torch_dtype=torch.float16).cuda().eval()
processor = CLIPProcessor.from_pretrained(model_id)
tokenizer = processor.tokenizer

pool = torch.arange(tokenizer.vocab_size, device="cuda")
for special in (tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id):
    pool = pool[pool != special]

image = load_image(image_url).convert("RGB")
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    image_features = model.get_image_features(**processor(images=image, return_tensors="pt").to("cuda")).float()
    image_features /= image_features.norm(dim=-1, keepdim=True)

def score(sequences):
    bos = torch.full((len(sequences), 1), tokenizer.bos_token_id, device="cuda")
    eos = torch.full((len(sequences), 1), tokenizer.eos_token_id, device="cuda")
    ids = torch.cat([bos, sequences, eos], dim=1)
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        features = model.get_text_features(input_ids=ids, attention_mask=torch.ones_like(ids)).float()
    features /= features.norm(dim=-1, keepdim=True)
    return (features @ image_features.T).squeeze(1)

sequences = pool[torch.randint(len(pool), (population, n_tokens), device="cuda")]
scores = score(sequences)

for generation in range(generations):
    elites = sequences[torch.argsort(scores, descending=True)[: population // 4]]
    parents = elites[torch.randint(len(elites), (population,))]
    mutation = torch.rand_like(parents, dtype=torch.float32) < 0.1
    parents[mutation] = pool[torch.randint(len(pool), (int(mutation.sum()),), device="cuda")]
    sequences, scores = parents, score(parents)
    print(f"{generation + 1:02d}: {float(scores.max()):.4f}")

best = sequences[scores.argmax()].tolist()
print("IDs:", best)
print("Decoded:", tokenizer.decode(best))